In [1]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chroma-db

In [2]:
from langchain_core.documents import Document 

In [3]:
sample_doc=Document(
    page_content="Hello From Piyush-Pranjal :)",
    metadata={"source":"https://www.google.com"}
)

In [4]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello From Piyush-Pranjal :)')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

In [6]:
# Load Text
from langchain_community.document_loaders.text import TextLoader

loader=TextLoader("data/python.txt",encoding="utf-8")

C:\Users\HP\AppData\Local\Temp\ipykernel_14152\2827052509.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [7]:
document=loader.load()

In [8]:
document

[Document(metadata={'source': 'data/python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [9]:
from langchain_community.document_loaders.pdf import PyPDFLoader

loader=PyPDFLoader("data/pdfs/research1.pdf")

document=loader.load()

document

[Document(metadata={'producer': 'pdfcpu v0.12.1 dev', 'creator': 'PyPDF', 'creationdate': '2026-07-29T06:50:27+00:00', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'book': 'Advances in Neural Information Processing Systems 30', 'created': '2017', 'date': '2017', 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallel

### Ingestion Pipeline 

In [10]:
# Data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [11]:
def load_all_documents():
    folder_path="data/pdfs"
    num_docs=0
    docs=[]

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path=os.path.join(folder_path,filename)
            
            loader=PyPDFLoader(pdf_path)
            all_docs=loader.load()
            docs.extend(all_docs)

            num_docs+=1

    print(f"Total docs:",num_docs)
    print(f"Total Pages:",len(docs))

    return docs



In [12]:
all_pdf_documents=load_all_documents()

Total docs: 2
Total Pages: 32


### Chunking 

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [14]:
def split_chunks(documents,chunk_size=500,chunk_overlap=50):

    splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunked_documents=splitter.split_documents(documents)

    return chunked_documents

In [15]:
chunks=split_chunks(all_pdf_documents)

In [16]:
len(chunks)

321

### Different Types of Chunking 

In [17]:
# 1. Fixed Size Chunking 

from langchain_text_splitters import CharacterTextSplitter

fixed_splitter=CharacterTextSplitter(
    separator=" ",
    chunk_size=500,
    chunk_overlap=50
)

fixed_chunks=fixed_splitter.split_documents(all_pdf_documents)

print(f"Number of chunks:",len(fixed_chunks))
fixed_chunks[0].page_content

Number of chunks: 330


'Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent'

In [18]:
# 2. Recursive Character Chunking 

from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["/n/n","/n"," ",""]
)

recursive_chunks=recursive_splitter.split_documents(all_pdf_documents)

print(f"Chunks:",len(recursive_chunks))
recursive_chunks[0].page_content

Chunks: 331


'Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent'

In [19]:
!pip install nltk

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\HP\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [20]:
# 3. Sentence Based Chunking 

import nltk 
from langchain_text_splitters import NLTKTextSplitter

nltk.download("punkt")
nltk.download("punkt_tab")

sentence_splitter=NLTKTextSplitter(
    chunk_size=500,
    chunk_overlap=50
    )

sentence_chunks=sentence_splitter.split_documents(
    all_pdf_documents
)

print(f"Chunks:",len(sentence_chunks))
print(sentence_chunks[0].page_content)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Created a chunk of size 569, which is longer than the specified 500
Created a chunk of size 570, which is longer than the specified 500
Created a chunk of size 631, which is longer than the specified 500
Created a chunk of size 620, which is longer than the specified 500
Created a chunk of size 898, which is longer than the specified 500


Chunks: 342
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder.


In [21]:
# 4. Paragraph Based Chunking

from langchain_text_splitters import RecursiveCharacterTextSplitter

paragraph_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0,
    separators=["/n/n","/n"," ",""]
)

paragraph_chunks=paragraph_splitter.split_documents(all_pdf_documents)

print(f"Chunks:",len(paragraph_chunks))

print(paragraph_chunks[0].page_content)

Chunks: 305
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent


In [22]:
# 5. Token based chunking 

from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

tokenizer=AutoTokenizer.from_pretrained("openai/gpt-oss-20b")

token_splitter=RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=500,
    chunk_overlap=50
)

token_chunks=token_splitter.split_documents(all_pdf_documents)

print(f"Chunks:",len(token_chunks))
print(token_chunks[0].page_content)

Chunks: 96
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring signiﬁcantly
less time to train. 

In [23]:
# %pip install -U langchain-experimental langchain-huggingface

In [24]:
# 6. Semantic Chunking 

from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

semantic_splitter=SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

semantic_chunks=semantic_splitter.split_documents(all_pdf_documents)

print(f"Chunks:",len(semantic_chunks))
print(semantic_chunks[0].page_content)

C:\Users\HP\AppData\Local\Temp\ipykernel_14152\699154258.py:3: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chunks: 152
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring signiﬁcantly
less time to train.

In [25]:
# 7. Structured based chunking 

from langchain_text_splitters import RecursiveCharacterTextSplitter

structure_splitter=RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=50,
    separators=["/n/n","/n"," ",""]
)

structure_chunks=[]

for doc in all_pdf_documents:
    structure_chunks.extend(
        structure_splitter.split_documents([doc])
    )

In [26]:
# 8. Parent-Child Chunking 

from langchain_text_splitters import RecursiveCharacterTextSplitter

parent_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

child_splitter=RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=30
)

parent_chunks=parent_splitter.split_documents(
    all_pdf_documents
)

child_chunks=[]

for parent in parent_chunks:
    child_chunks.extend(child_splitter.split_documents([parent]))

print(f"Parent Chunk:",len(parent_chunks))
print(f"Child Chunk:",len(child_chunks))

Parent Chunk: 169
Child Chunk: 766


### Embeddings

In [27]:
from sentence_transformers import SentenceTransformer

In [28]:
class EmbeddingManager():
    def __init__(self,model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        print("Loading model....")
        self.model=SentenceTransformer(self.model_name)
        print("Dimensions:",self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self,text):
        embeddings=self.model.encode(text,show_progress_bar=True)
        print("embeddings shape:",embeddings.shape)
        return embeddings


In [29]:
embeddingmanager=EmbeddingManager()

Loading model....


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Dimensions: 384


C:\Users\HP\AppData\Local\Temp\ipykernel_14152\1991556102.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Dimensions:",self.model.get_sentence_embedding_dimension())


### Vector Store

In [30]:
import chromadb
import uuid

In [31]:
class VectorStoreManager:
    def __init__(self,persist_directory="data/vector_store",collection_name="pdf_documents"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.collection=None
        self.client=None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory,exist_ok=True)

        # Create a client

        self.client=chromadb.PersistentClient(path=self.persist_directory)

        # Create a collection

        self.collection=self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description":"Vector Store for pdf embeddings in RAG"
            }
        )

        print(f"Initialized the vector store with collection :",self.collection_name)
        print(f"Docs in collection:",self.collection.count())

    def add_documents(self,documents,embeddings):
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents does not match number of embeddings")

        ids=[]
        all_metadata=[]
        documents_content=[]
        embedding_list=[]

        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata["doc_index"]=i
            metadata["content_length"]=len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            embedding_list.append(embedding.tolist())

        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embedding_list
        )

        print(f"total documents added in vector stores:",len(documents_content))
        print(f"docs in collection",self.collection.count())

In [32]:
vector_store=VectorStoreManager()

Initialized the vector store with collection : pdf_documents
Docs in collection: 321


In [33]:
text=[doc.page_content for doc in chunks]

embeddings=embeddingmanager.generate_embeddings(text)

vector_store.add_documents(chunks,embeddings)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embeddings shape: (321, 384)
total documents added in vector stores: 321
docs in collection 642


### Retriever

In [45]:
from sklearn.metrics.pairwise import cosine_similarity

In [46]:
class RAGRetriever:
    def __init__(self,embedding_manager,vector_store):
        self.embedding_manager=embedding_manager
        self.vector_store=vector_store


    def retrieve(self,query,top_k=5,score_threshold=0.0):
        query_embeddings=self.embedding_manager.generate_embeddings([query])[0]

        results=self.vector_store.collection.query(
            query_embeddings=query_embeddings,
            n_results=top_k

        )
        retrieved_docs=[]

        if results['documents'] and results['documents'][0]:
            ids=results['ids'][0]
            metadatas=results['metadatas'][0]
            documents=results['documents'][0]
            distances=results['distances'][0]

            for i,(doc_id,metadata,document,distance) in enumerate(zip(ids,metadatas,documents,distances)):

                similarity_score=1-distance

                if similarity_score>=score_threshold:
                    retrieved_docs.append({
                        "id":doc_id,
                        "metadata":metadata,
                        "document":document,
                        "distance":distance,
                        "simalarity_score":similarity_score,
                        "rank":i+1
                    })

            print(f"Retrieved {len(retrieved_docs)} documents")
        else: 
            print("No valid data")


        return retrieved_docs

In [47]:
rag_retriever=RAGRetriever(embeddingmanager,vector_store)

In [48]:
rag_retriever.retrieve("What is RAG?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
Retrieved 5 documents


[{'id': 'doc_b9ff6891-148a-4ac3-826f-039624f86507',
  'metadata': {'doc_index': 88,
   'producer': 'pdfTeX-1.40.25',
   'page': 0,
   'total_pages': 21,
   'author': '',
   'content_length': 288,
   'title': '',
   'moddate': '2024-03-28T00:54:45+00:00',
   'creationdate': '2024-03-28T00:54:45+00:00',
   'source': 'data/pdfs\\research2.pdf',
   'subject': '',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'page_label': '1',
   'keywords': '',
   'trapped': '/False',
   'creator': 'LaTeX with hyperref'},
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'distance': 0.46291473507881165,
  'simalarity_score': 0.5370852649211884,
  'rank': 1},
 {'id': 'doc_b62da2af

In [49]:
rag_retriever.retrieve("What is encoder-decoder?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
Retrieved 5 documents


[{'id': 'doc_6def3e3a-7d58-4e94-beaa-a1cbb1f71585',
  'metadata': {'creationdate': '2026-07-29T06:50:27+00:00',
   'page_label': '3',
   'published': '2017',
   'source': 'data/pdfs\\research1.pdf',
   'created': '2017',
   'subject': 'Neural Information Processing Systems http://nips.cc/',
   'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existin

### Generation

In [53]:
# !pip install langchain-groq

In [76]:
from dotenv import load_dotenv
import os 

load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

In [77]:
from langchain_groq import ChatGroq

llm=ChatGroq(
    groq_api_key=groq_api_key,
    model="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024
)

In [78]:
def generate_output(query,retriever,llm,top_k=3):

    results=retriever.retrieve(query,top_k)

    context="/n".join([doc['document'] for doc in results])

    if not context:
        print("We found no relevant context for the query")

    prompt=f""" Use given context to generate the answer for the query,
            query:{query},
            context:{context}
            
            """

    response=llm.invoke([prompt.format(context=context,query=query)])

    return response.content

In [79]:
answer=generate_output("What is encoder decoder",rag_retriever,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
Retrieved 3 documents


In [80]:
answer

'**Encoder‑decoder** is a neural‑network architecture that splits the processing of a sequence into two distinct stages:\n\n| Stage | Purpose | Key components (from the context) |\n|-------|---------|-----------------------------------|\n| **Encoder** | Transforms the input sequence into a set of continuous representations that capture its meaning. | • 6 identical layers (N\u202f=\u202f6) <br>• Each layer has two sub‑layers: self‑attention and a position‑wise feed‑forward network <br>• Residual connections + layer‑norm around each sub‑layer <br>• Positional encodings added to the input embeddings <br>• Drop‑out rate \\(P_{\\text{drop}} = 0.1\\) <br>• Output dimension \\(d_{\\text{model}} = 512\\) |\n| **Decoder** | Generates the output sequence one token at a time, conditioning on the encoder’s representation and on the tokens it has already produced. | • 6 identical layers (N\u202f=\u202f6) <br>• Each layer has three sub‑layers: <br>\u20031. Masked self‑attention over the decoder’s ow

In [83]:
from groq import Groq

client=Groq(api_key=groq_api_key)

models=client.models.list()

for model in models.data:
    print(model.id)

whisper-large-v3-turbo
allam-2-7b
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-arabic-saudi
qwen/qwen3.8-27b
whisper-large-v3
openai/gpt-oss-120b
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-v1-english


### BM25 

In [89]:
# !pip install rank-bm25

In [90]:
from rank_bm25 import BM25Okapi

bm25_documents=[
    doc.page_content for doc in all_pdf_documents
]

tokenized_documents=[
    doc.lower().split() for doc in bm25_documents
]

bm25=BM25Okapi(tokenized_documents)

print("Total documents indexed:",len(tokenized_documents))

Total documents indexed: 32


In [91]:
def bm25_retrieve(query,top_k=5):
    tokenized_query=query.lower().split()

    scores=bm25.get_scores(tokenized_query)

    ranked_indices=scores.argsort()[::-1][:top_k]

    results=[]


    for rank,idx in enumerate(ranked_indices,start=1):
        results.append(
            {
                "document":bm25_documents[idx],
                "score":float(scores[idx]),
                "rank":rank
            }
        )

        return results

In [92]:
results=bm25_retrieve("What is encoder decoder")

In [94]:
for result in results:
    print(f"\nRank: {result['rank']}")
    print(f"Score: {result['score']:.4f}")
    print(result["document"][:500])
    print("-" * 50)


Rank: 1
Score: 5.7850
MultiHead(Q,K,V ) = Concat(head 1,..., headh)W O
where headi = Attention(QW Q
i ,KW K
i ,VW V
i )
Where the projections are parameter matricesW Q
i ∈ Rdmodel×dk,W K
i ∈ Rdmodel×dk,W V
i ∈ Rdmodel×dv
andW O∈ Rhdv×dmodel.
In this work we employ h = 8 parallel attention layers, or heads. For each of these we use
dk =dv =dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost
is similar to that of single-head attention with full dimensionality.
3.2.3 Applications of At
--------------------------------------------------


### Hybrid Retrieval

In [96]:
def reciprocal_rank_fusion(result_lists, k=60):
    fused_scores = {}
    documents = {}

    for results in result_lists:
        for result in results:
            doc = result["document"]

            # Document text as the common identifier
            doc_id = doc if isinstance(doc, str) else doc.strip()

            if doc_id not in fused_scores:
                fused_scores[doc_id] = 0.0
                documents[doc_id] = result

            fused_scores[doc_id] += 1 / (k + result["rank"])

    ranked_docs = sorted(
        fused_scores.items(),
        key=lambda item: item[1],
        reverse=True
    )

    fused_results = []

    for rank, (doc_id, score) in enumerate(ranked_docs, start=1):
        result = documents[doc_id].copy()
        result["rrf_score"] = score
        result["rank"] = rank
        fused_results.append(result)

    return fused_results

In [97]:
def dense_retrieve_for_rrf(query, top_k=5):
    results = rag_retriever.retrieve(query, top_k=top_k)

    return [
        {
            "document": result["document"],
            "rank": result["rank"]
        }
        for result in results
    ]

In [98]:
def hybrid_retrieve(query, top_k=5, candidate_k=10):
    bm25_results = bm25_retrieve(query, top_k=candidate_k)

    dense_results = dense_retrieve_for_rrf(
        query,
        top_k=candidate_k
    )

    fused_results = reciprocal_rank_fusion(
        [bm25_results, dense_results]
    )

    return fused_results[:top_k]

In [99]:
results = hybrid_retrieve(
    "What is encoder decoder?",
    top_k=5,
    candidate_k=10
)

for result in results:
    print(f"\nRank: {result['rank']}")
    print(f"RRF Score: {result['rrf_score']:.5f}")
    print(result["document"][:500])
    print("-" * 50)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
Retrieved 10 documents

Rank: 1
RRF Score: 0.03252
layers, produce outputs of dimensiondmodel = 512.
Decoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two
sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head
attention over the output of the encoder stack. Similar to the encoder, we employ residual connections
around each of the sub-layers, followed by layer normalization. We also modify the self-attention
--------------------------------------------------

Rank: 2
RRF Score: 0.03150
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The ﬁrst is a multi-head self-attention mechanism, and the second is a simple, position-
2
piyushthakur1611@gmail.com
--------------------------------------------------

Rank: 3
RRF Score: 0.03054
positional encodings in both the encoder and decode

### Cross Encoder Reranking 

In [101]:
from sentence_transformers import CrossEncoder 

cross_encoder=CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [102]:
def rerank_documents(query, documents, top_k=5):
    if not documents:
        return []

    pairs = [
        (query, result["document"])
        for result in documents
    ]

    scores = cross_encoder.predict(pairs)

    reranked_results = []

    for result, score in zip(documents, scores):
        item = result.copy()
        item["rerank_score"] = float(score)
        reranked_results.append(item)

    reranked_results.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return reranked_results[:top_k]

In [103]:
query = "What is encoder decoder?"

# Retrieve candidates using BM25 + dense retrieval + RRF
hybrid_results = hybrid_retrieve(
    query,
    top_k=10,
    candidate_k=20
)

# Rerank candidates using Cross-Encoder
final_results = rerank_documents(
    query,
    hybrid_results,
    top_k=5
)

for result in final_results:
    print(f"\nRerank score: {result['rerank_score']:.4f}")
    print(result["document"][:500])
    print("-" * 50)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
Retrieved 10 documents

Rerank score: 5.0195
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The ﬁrst is a multi-head self-attention mechanism, and the second is a simple, position-
2
piyushthakur1611@gmail.com
--------------------------------------------------

Rerank score: 4.5167
layers, produce outputs of dimensiondmodel = 512.
Decoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two
sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head
attention over the output of the encoder stack. Similar to the encoder, we employ residual connections
around each of the sub-layers, followed by layer normalization. We also modify the self-attention
--------------------------------------------------

Rerank score: 3.5292
[31, 2, 8].
• The encoder contains self-attention layers. In a self-a